In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2021_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2021-01.csv", sep=";",low_memory=False)

In [3]:
df_2021_02 = pd.read_csv(
    "/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2021-02.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

In [4]:
df_2021_01.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA,01/01/2021,"4,599",NaN,R$ / litro,BRANCA
1,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,ETANOL,01/01/2021,"4,199",NaN,R$ / litro,BRANCA
2,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA ADITIVADA,01/01/2021,"4,799",NaN,R$ / litro,BRANCA
3,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL,01/01/2021,"3,499",NaN,R$ / litro,BRANCA
4,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL S10,01/01/2021,"3,599",NaN,R$ / litro,BRANCA


In [5]:
df_2021_02.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,NE,CE,MARACANAU,BEZERRA & MENDES COMERCIAL DE PETRÓLEO LTDA.,05.397.086/0001-51,RODOVIA MARANGUAPE - FORTALEZA - KM 06,S/N,NaN,PARQUE LUZARDO VIANA,61910-000,GASOLINA ADITIVADA,01/07/2021,"5,699",NaN,R$ / litro,IPIRANGA
1,NE,CE,MARACANAU,BEZERRA & MENDES COMERCIAL DE PETRÓLEO LTDA.,05.397.086/0001-51,RODOVIA MARANGUAPE - FORTALEZA - KM 06,S/N,NaN,PARQUE LUZARDO VIANA,61910-000,GASOLINA,01/07/2021,"5,499",NaN,R$ / litro,IPIRANGA
2,NE,CE,MARACANAU,BEZERRA & MENDES COMERCIAL DE PETRÓLEO LTDA.,05.397.086/0001-51,RODOVIA MARANGUAPE - FORTALEZA - KM 06,S/N,NaN,PARQUE LUZARDO VIANA,61910-000,DIESEL S10,01/07/2021,"4,699",NaN,R$ / litro,IPIRANGA
3,NE,CE,MARACANAU,BEZERRA & MENDES COMERCIAL DE PETRÓLEO LTDA.,05.397.086/0001-51,RODOVIA MARANGUAPE - FORTALEZA - KM 06,S/N,NaN,PARQUE LUZARDO VIANA,61910-000,ETANOL,01/07/2021,"5,2",NaN,R$ / litro,IPIRANGA
4,NE,CE,MARACANAU,LUIZA GLAURIA R T MENEZES,03.602.329/0001-10,ESTRADA FORTALEZA MARANGUAPE,S/N,NaN,KAGADO,61901-410,GASOLINA,01/07/2021,"5,49",NaN,R$ / litro,VIBRA ENERGIA


In [6]:
df_2021 = pd.concat((df_2021_01, df_2021_02))

In [7]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 807537 entries, 0 to 472855
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     807537 non-null  object 
 1   Estado - Sigla     807537 non-null  object 
 2   Municipio          807537 non-null  object 
 3   Revenda            807537 non-null  object 
 4   CNPJ da Revenda    807537 non-null  object 
 5   Nome da Rua        807537 non-null  object 
 6   Numero Rua         807222 non-null  object 
 7   Complemento        165365 non-null  object 
 8   Bairro             805808 non-null  object 
 9   Cep                807537 non-null  object 
 10  Produto            807537 non-null  object 
 11  Data da Coleta     807537 non-null  object 
 12  Valor de Venda     807537 non-null  object 
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  807537 non-null  object 
 15  Bandeira           807537 non-null  object 
dtypes: floa

In [8]:
df_2021_pe = df_2021[df_2021["Estado - Sigla"] == "PE"]

In [9]:
df_2021_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
1658,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,DIESEL S10,05/01/2021,"3,59",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1659,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,GNV,05/01/2021,"2,939",NaN,R$ / m³,PETROBRAS DISTRIBUIDORA S.A.
1660,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,ETANOL,05/01/2021,"3,59",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1661,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,GASOLINA,05/01/2021,"4,69",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1662,NE,PE,JABOATAO DOS GUARARAPES,C.E.S BELTRAO COMERCIO DE COMBUSTIVEIS LTDA,06.187.301/0001-52,RODOVIA PE 07,S/N,KM 21 625,VARGEM FRIA,54125-130,ETANOL,03/01/2021,"3,39",NaN,R$ / litro,BRANCA


In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2021")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [11]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2021_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
